In [1]:
import pystac_client
import planetary_computer
import xarray as xr
import rasterio as rio
import rioxarray
import geopandas as gpd
import odc.stac
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import os
import rasterstats
from rioxarray import merge
import scipy.ndimage as ndimage
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.measure import label
from shapely.geometry import shape
from rasterio.features import shapes, rasterize
import xyzservices.providers as xyz
from datetime import datetime
import datetime

In [2]:
# Connect to Planetary Computer STAC
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

# Mui Ca Mau bounding box
bbox = [104.7, 8.55, 104.9, 8.7]

In [3]:
#Choose dry season months
def filter_dry_season(items, dry_season_months=[12, 1, 2, 3, 4]):
    filtered = [
        item for item in items
        if item.datetime.month in dry_season_months
    ]
    print(f"  {len(items)} total → {len(filtered)} dry season scenes")
    return filtered

In [4]:
#Find dry season images from 2016 and 2017- had to include this because 2016 alone didn't have enough images
search_2017 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2015-11-01/2017-04-30",  # spans two dry seasons
    query={"eo:cloud_cover": {"lt": 25}}
)

items_2017_all = list(search_2017.items())
items_2017 = filter_dry_season(items_2017_all)

  7 total → 6 dry season scenes


In [5]:
search_2026 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-11-01/2026-04-30",
    query={"eo:cloud_cover": {"lt": 25}}
)

items_2026 = list(search_2026.items())
print(f"Found {len(items_2026)} scenes for 2026")

Found 7 scenes for 2026


In [6]:
items_2017_48PVQ = [
    item for item in items_2017
    if item.properties["s2:mgrs_tile"] == "48PVQ"
]

items_2026_48PVQ = [
    item for item in items_2026
    if item.properties["s2:mgrs_tile"] == "48PVQ"
]

print(f"2017: {len(items_2017_48PVQ)} scenes")
print(f"2026: {len(items_2026_48PVQ)} scenes")

2017: 6 scenes
2026: 7 scenes


In [7]:
bands = ["B03", "B04", "B08", "B11", "SCL"]  #  Green, Red, NIR,SWIR, SLC

In [8]:
camau_2017_s2_ds = odc.stac.load(
    items_2017_48PVQ,
    bands=bands,
    chunks={"x": 256, "y": 256},
    groupby="solar_day",
    bbox=bbox,
    crs="EPSG:32648",
    resolution=10
)

camau_2026_s2_ds = odc.stac.load(
    items_2026_48PVQ,
    bands=bands,
    chunks={"x": 256, "y": 256},
    groupby="solar_day",
    bbox=bbox,
    crs="EPSG:32648",
    resolution=10
)

In [9]:
camau_2026_s2_da = camau_2026_s2_ds.to_array(dim='band')

In [10]:
def harmonize_to_old(data):
    """
    Harmonize new Sentinel-2 data to the old baseline.

    Parameters
    ----------
    data: xarray.DataArray
        A DataArray with four dimensions: time, band, y, x

    Returns
    -------
    harmonized: xarray.DataArray
        A DataArray with all values harmonized to the old
        processing baseline.
    """

    cutoff = datetime.datetime(2022, 1, 25)
    offset = 1000
    bands = [
        "B01",
        "B02",
        "B03",
        "B04",
        "B05",
        "B06",
        "B07",
        "B08",
        "B8A",
        "B09",
        "B10",
        "B11",
        "B12",
    ]

    old = data.sel(time=slice(cutoff))

    to_process = list(set(bands) & set(data.band.data.tolist()))
    new = data.sel(time=slice(cutoff, None)).drop_sel(band=to_process)

    new_harmonized = data.sel(time=slice(cutoff, None), band=to_process).clip(offset)
    new_harmonized -= offset

    new = xr.concat([new, new_harmonized], "band").sel(band=data.band.data.tolist())
    return xr.concat([old, new], dim="time")

In [11]:
camau_2026_s2_da_harmonized = harmonize_to_old(camau_2026_s2_da)
camau_2026_s2_ds = camau_2026_s2_da_harmonized.to_dataset(dim='band')

In [12]:
def mask_and_scale(ds):
    bad_scl = [0, 1, 8, 9]
    valid = ~ds["SCL"].isin(bad_scl)
    
    spectral_bands = ["B03", "B04", "B08", "B11"]
    masked = ds[spectral_bands].where(valid) 
    return masked

masked_2017 = mask_and_scale(camau_2017_s2_ds)
masked_2026 = mask_and_scale(camau_2026_s2_ds)

In [13]:
# Median composite
comp_2017 = masked_2017.median(dim="time", skipna=True).compute()
comp_2026 = masked_2026.median(dim="time", skipna=True).compute()

/opt/conda/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


In [14]:
comp_2026

<xarray.Dataset> Size: 59MB
Dimensions:      (y: 1660, x: 2202)
Coordinates:
  * y            (y) float64 13kB 9.617e+05 9.617e+05 ... 9.451e+05 9.451e+05
  * x            (x) float64 18kB 4.67e+05 4.67e+05 ... 4.89e+05 4.89e+05
    spatial_ref  int32 4B 32648
Data variables:
    B03          (y, x) float32 15MB 1.132e+03 1.138e+03 ... 1.194e+03 1.206e+03
    B04          (y, x) float32 15MB 1.386e+03 1.378e+03 ... 1.38e+03 1.407e+03
    B08          (y, x) float32 15MB 844.0 842.0 850.0 ... 693.5 681.0 705.5
    B11          (y, x) float32 15MB 214.0 214.0 220.0 ... 222.0 222.0 222.0

In [15]:
comp_2017

<xarray.Dataset> Size: 59MB
Dimensions:      (y: 1660, x: 2202)
Coordinates:
  * y            (y) float64 13kB 9.617e+05 9.617e+05 ... 9.451e+05 9.451e+05
  * x            (x) float64 18kB 4.67e+05 4.67e+05 ... 4.89e+05 4.89e+05
    spatial_ref  int32 4B 32648
Data variables:
    B03          (y, x) float32 15MB 1.146e+03 1.138e+03 ... 1.338e+03 1.282e+03
    B04          (y, x) float32 15MB 1.122e+03 1.133e+03 ... 1.208e+03 1.207e+03
    B08          (y, x) float32 15MB 544.5 549.0 535.5 ... 613.0 619.0 585.5
    B11          (y, x) float32 15MB 272.5 272.5 282.5 ... 401.5 393.0 393.0

In [16]:
#And indices function
def add_indices(ds):
    G   = ds["B03"]
    R   = ds["B04"]
    NIR = ds["B08"]
    S1  = ds['B11']

    ds["NDVI"]  = (NIR - R)   / (NIR + R)
    ds["NDWI"]  = (G - NIR)   / (G + NIR)
    ds["CMRI"]  = ds["NDVI"] - ds["NDWI"]
    return ds.compute()

In [17]:
comp_2017 = add_indices(comp_2017)
comp_2026 = add_indices(comp_2026)

In [19]:
for da, year in [(comp_2017, "2017"), (comp_2026, "2026")]:
    da.rio.write_crs("EPSG:32648", inplace=True)
    da.rio.to_raster(f"ca_mau_{year}_48PVQ.tif")
    print(f"Exported ca_mau_{year}_48PVQ.tif")

Exported ca_mau_2017_48PVQ.tif
Exported ca_mau_2026_48PVQ.tif
